# 03 — Train the tooth detector (32 FDI classes)

Kaggle GPU notebook. Rebuilds the **identical** split used locally (seed 42)
from the public DENTEX dataset, so the val numbers are comparable to
`02_prepare_dataset.ipynb`.

**Why this replaces the coordinator + RF:** the old pipeline found four
quadrant boxes and then guessed the tooth number from lesion-box geometry
(75.9% accuracy). This model localises every tooth and labels it with its FDI
number directly.

**Augmentation note — flips are disabled.** FDI labels are position-dependent:
mirroring a panorama turns an upper-right tooth into an upper-left one, so
`fliplr`/`flipud`/`mosaic` would inject wrong labels. Only small rotation,
scale and translation are used.

Expected: 540 train / 94 val images, 18,095 tooth boxes.

In [ ]:
try:
    import ultralytics
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "ultralytics"], check=True)
    import ultralytics

import torch
print("ultralytics", ultralytics.__version__)
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print("gpu:", p.name, f"{p.total_memory / 1e9:.1f} GB")

In [ ]:
import glob
import json
import random
from collections import Counter, defaultdict
from pathlib import Path

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
SEED, VAL_FRAC = 42, 0.15

DISEASE_NAMES = ["Impacted", "Caries", "Periapical Lesion", "Deep Caries"]


def load_coco(path):
    with open(path) as fh:
        return json.load(fh)


def image_map(coco):
    return {i["id"]: i for i in coco["images"]}


def yolo_labels(coco, class_fn):
    imgs = image_map(coco)
    out = defaultdict(list)
    for ann in coco["annotations"]:
        img = imgs[ann["image_id"]]
        W, H = img["width"], img["height"]
        x, y, w, h = ann["bbox"]
        x1, y1 = max(0.0, x), max(0.0, y)
        x2, y2 = min(W, x + w), min(H, y + h)
        w, h = x2 - x1, y2 - y1
        if w <= 1 or h <= 1:
            continue
        out[img["file_name"]].append(
            f"{class_fn(ann)} {(x1 + w / 2) / W:.6f} {(y1 + h / 2) / H:.6f} {w / W:.6f} {h / H:.6f}")
    return dict(out)


def stratified_split(names, signature_fn, val_frac=VAL_FRAC, seed=SEED):
    groups = defaultdict(list)
    for n in names:
        groups[signature_fn(n)].append(n)
    rng = random.Random(seed)
    train, val = [], []
    for key in sorted(groups, key=str):
        members = sorted(groups[key])
        rng.shuffle(members)
        n_val = max(1, round(len(members) * val_frac)) if len(members) > 1 else 0
        val.extend(members[:n_val])
        train.extend(members[n_val:])
    return sorted(train), sorted(val)


def build(out_root, name, image_src, labels, train, val, class_names):
    root = Path(out_root) / name
    for split in ("train", "val"):
        (root / "images" / split).mkdir(parents=True, exist_ok=True)
        (root / "labels" / split).mkdir(parents=True, exist_ok=True)
    for split, names in (("train", train), ("val", val)):
        for fname in names:
            src = Path(image_src) / fname
            dst = root / "images" / split / fname
            if not dst.exists():
                dst.symlink_to(src)
            rows = labels.get(fname, [])
            (root / "labels" / split / (Path(fname).stem + ".txt")).write_text(
                "\n".join(rows) + ("\n" if rows else ""))
    yaml = f"path: {root.resolve()}\ntrain: images/train\nval: images/val\nnames:\n"
    yaml += "".join(f"  {i}: {n}\n" for i, n in enumerate(class_names))
    (root / "data.yaml").write_text(yaml)
    return root

In [ ]:
# ---- locate the public DENTEX dataset and build the tooth dataset ----
qe_json = next(INPUT.rglob("train_quadrant_enumeration.json"))
qe = load_coco(qe_json)
qe_imgs = qe_json.parent / "xrays"
print("annotations:", qe_json)
print("images dir :", qe_imgs, "|", len(list(qe_imgs.glob('*.png'))), "png files")
print("images in json:", len(qe["images"]), "| boxes:", len(qe["annotations"]))

tooth_names = [f"FDI_{q}{t}" for q in range(1, 5) for t in range(1, 9)]
labels = yolo_labels(qe, lambda a: (a["category_id_1"]) * 8 + a["category_id_2"])

def signature(name):
    n = len(labels.get(name, []))
    return "sparse" if n < 20 else ("dense" if n > 30 else "normal")

train_names, val_names = stratified_split([i["file_name"] for i in qe["images"]], signature)
root = build(WORK / "datasets", "tooth", qe_imgs, labels, train_names, val_names, tooth_names)

n_train_boxes = sum(len(labels.get(n, [])) for n in train_names)
n_val_boxes = sum(len(labels.get(n, [])) for n in val_names)
print(f"train {len(train_names)} imgs / {n_train_boxes} boxes")
print(f"val   {len(val_names)} imgs / {n_val_boxes} boxes")
assert len(train_names) == 540 and len(val_names) == 94, "split drifted from the local build"
assert n_train_boxes + n_val_boxes == 18095, "box count drifted"
print("split matches the local build exactly.")

In [ ]:
from ultralytics import YOLO

IMGSZ = 1024          # ~2.8x downscale of the 2872 px panoramas
EPOCHS = 100
BATCH = 8             # T4/P100 16 GB; drop to 4 if OOM
MODEL = "yolov8s.pt"

model = YOLO(MODEL)
results = model.train(
    data=str(root / "data.yaml"),
    imgsz=IMGSZ,
    epochs=EPOCHS,
    batch=BATCH,
    patience=25,
    seed=SEED,
    deterministic=True,
    optimizer="auto",
    cos_lr=True,
    close_mosaic=10,
    amp=True,
    workers=2,
    project=str(WORK / "runs"),
    name="tooth_v2",
    exist_ok=True,
    # position-dependent labels: no flips, no mosaic
    fliplr=0.0,
    flipud=0.0,
    mosaic=0.0,
    degrees=3.0,
    translate=0.05,
    scale=0.15,
    shear=0.0,
    perspective=0.0,
    hsv_h=0.015,
    hsv_s=0.3,
    hsv_v=0.3,
)
print("best weights:", results.save_dir)

In [ ]:
# ---- validation metrics ----
m = model.val(data=str(root / "data.yaml"), imgsz=IMGSZ, split="val", conf=0.001, iou=0.6)
names = [m.names[i] for i in sorted(m.names)]
ap50 = m.box.ap50.tolist()

print(f"mAP50    {m.box.map50:.4f}")
print(f"mAP50-95 {m.box.map:.4f}")
print(f"precision {m.box.mp:.4f} | recall {m.box.mr:.4f}")
print()
for name, ap in sorted(zip(names, ap50), key=lambda t: -t[1]):
    print(f"  {name:10s} AP50 {ap:.3f}")

summary = {
    "model": MODEL, "imgsz": IMGSZ, "epochs": EPOCHS, "batch": BATCH,
    "train_images": len(train_names), "val_images": len(val_names),
    "mAP50": float(m.box.map50), "mAP50_95": float(m.box.map),
    "precision": float(m.box.mp), "recall": float(m.box.mr),
    "ap50_per_class": dict(zip(names, map(float, ap50))),
}
(WORK / "tooth_metrics.json").write_text(json.dumps(summary, indent=2))
print("\nwrote tooth_metrics.json")

In [ ]:
# ---- package outputs for download ----
import shutil
best = Path(results.save_dir) / "weights" / "best.pt"
shutil.copy2(best, WORK / "tooth_best.pt")
shutil.make_archive(str(WORK / "tooth_run"), "zip", str(results.save_dir))
print("tooth_best.pt:", (WORK / "tooth_best.pt").stat().st_size / 1e6, "MB")
print("artifacts in /kaggle/working: tooth_best.pt, tooth_metrics.json, tooth_run.zip")

## What to do with the output

Download `tooth_best.pt` + `tooth_metrics.json` from the notebook output and
drop the weights in `dentex_week3/production/` (gitignored). The next notebook
(`04_train_disease_detector.ipynb`) trains the lesion detector; `05` joins the
two by IoU and evaluates end-to-end.

Compare against the v1 baseline on the **locked 50-image set**, not on this
val split: v1 tooth accuracy was 75.9%.